# MDAseMaker tutorial

This notebook demonstrates `MDAseMaker`, which runs an MD producing an MD.traj file for each run.

## NVT Simulation

In [ ]:
# Imports - adapt as needed for your environment

from ase.lattice.cubic import Diamond
from jobflow import run_locally
from pymatgen.io.ase import AseAtomsAdaptor

from autoplex.data.md.flows import MDAseMaker

The following sets up the MD and runs it locally.

In [ ]:
# MD example - adapt model path and pre_database_dir
atoms = Diamond(symbol="Si", latticeconstant=5.43)
pmg_structure = AseAtomsAdaptor.get_structure(atoms)

md_job = MDAseMaker(
    force_field_name="MLFF.MACE_MPA_0",
    calculator_kwargs={"device": "cpu"},
    temperature_list=[300.0, 500.0, 400.0],
    eqm_step_list=[3,3,3],
    rate_list=[0,2],
    time_step=1.0,
    traj_interval=1,
    ensemble="nvt",
    dynamics="langevin",
    traj_file="MD.traj",
    traj_file_fmt="ase",
    store_trajectory="partial",
).make(structure=pmg_structure)

run_locally(md_job,
           create_folders=False,
           ensure_success=True)

print("MD workflow submitted")

### Temperature profile visualization
Plot the temperature schedule produced by `generate_temperature_profile`. After running, you should see an image named `md_temperature_profile.png` saved in the current directory.

In [ ]:
from autoplex.data.md.utils import generate_temperature_profile, plot_temperature_profile

temp_list = [300., 500.0, 400.0]
eqm_steps = [3, 3, 3] #Nr of equilibration steps at each T
rates = [0, 2] #for ramping between 300->500 K and 500->400 K, 0 equals no ramping, smaller values mean faster change

T_array, n_steps = generate_temperature_profile(temp_list, eqm_steps, rates)
plot_temperature_profile(T_array, time_step=1.0)  # writes md_temperature_profile.png

## NPT Simulation

### MD at constant pressure (NPT)

The run above is NVT: the cell stays fixed. With `ensemble="npt"` a barostat lets
the cell change so that the pressure stays at the target value. NPT needs three
more settings:

- `pressure`: target pressure in kbar (`0.0` is close to ambient; 1 bar = 0.001 kbar).
  A list of pressures is ramped evenly over the whole run.
- `dynamics`: `"nose-hoover-chain"` (atomate2's default for NPT, ASE `MTKNPT`), or
  `"berendsen"` for quick equilibration (it does not sample a correct NPT ensemble).
- `ase_md_kwargs`: damping times in ASE units. A good start is about 100 time steps for
  the thermostat (`tdamp`) and about 1000 time steps for the barostat (`pdamp`).

The temperature profile (`temperature_list`, `eqm_step_list`, `rate_list`) works
the same as in NVT.

In [ ]:
from ase import units

md_job_npt = MDAseMaker(
    force_field_name="MLFF.MACE_MPA_0",
    calculator_kwargs={"device": "cpu"},
    temperature_list=[300.0, 500.0, 400.0],
    eqm_step_list=[3,3,3],
    rate_list=[0, 2],
    time_step=1.0,
    traj_interval=1,
    ensemble="npt",
    dynamics="nose-hoover",  # ASE NPT: works with every atomate2 version used in CI
    pressure=0.0,  # kbar
    ase_md_kwargs={
        "ttime": 25 * units.fs,  # thermostat time constant
        "pfactor": (75 * units.fs) ** 2 * 100 * units.GPa,  # (barostat time)^2 x bulk modulus of Si
    },
    traj_file="MD_npt.traj",  # own file, so the NVT trajectory MD.traj is kept
    traj_file_fmt="ase",
    store_trajectory="partial",
).make(structure=pmg_structure)

run_locally(md_job_npt, create_folders=False, ensure_success=True)

### Compare the volumes from NVT and NPT

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ase.io import read

time_step = 1.0  # fs, as in both MD runs

fig, ax = plt.subplots(figsize=(4, 2.5))
for traj_file, label, color in [
    ("MD.traj", "NVT", "#2a78d6"),
    ("MD_npt.traj", "NPT", "#eb6834"),
]:
    frames = read(traj_file, index=":")
    volume = np.array([atoms.get_volume() for atoms in frames])
    time_ps = np.arange(len(volume)) * time_step / 1000
    ax.plot(time_ps, volume, color=color, linewidth=1.5, label=label)

ax.set_xlabel("Simulation time (ps)")
ax.set_ylabel("Cell volume (Å$^3$)")
ax.grid(visible=True, linestyle="--", linewidth=0.5, alpha=1)
ax.legend(frameon=False)
plt.show()

Note that for testing reasons the runs are very short. Unlike NVT, the NPT volume changes: the cell starts at the experimental lattice constant (5.43 Å), which is about 2 % too small in volume for MACE-MPA-0 (equilibrium 5.47 Å), so the barostat expands it, overshoots and swings back. With only 8 atoms the volume also fluctuates by about 2 % in equilibrium. For production runs, start from a relaxed cell, use a larger supercell, and run several ps before averaging.